# 📝 TF-IDF Experiment: Text Feature Engineering

**Project:** Gemma 4 - Good AI for Humanity, Not Just Performance  
**Notebook:** TF-IDF based feature engineering for quality prediction  
**Author:** Data Science Team  
**Last Updated:** 2024-02-25

## 📋 Table of Contents
1. [Introduction](#introduction)
2. [Data Loading](#data-loading)
3. [TF-IDF on Prompts](#tfidf-prompts)
4. [TF-IDF on Responses](#tfidf-responses)
5. [Combined TF-IDF Features](#combined-tfidf)
6. [N-Gram Analysis](#ngram-analysis)
7. [Model Training with TF-IDF](#model-training)
8. [Feature Importance](#feature-importance)
9. [Results Comparison](#results-comparison)

---
## 1. Introduction <a name="introduction"></a>

This notebook experiments with TF-IDF (Term Frequency-Inverse Document Frequency) features for improving quality score prediction. TF-IDF captures the importance of words relative to the entire corpus.

**Hypotheses:**
1. Certain vocabulary patterns correlate with higher quality scores
2. Response TF-IDF features will be more predictive than prompt features
3. N-gram features (bigrams, trigrams) will capture structural patterns

In [ ]:
# ============================================
# Cell 1: Imports
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded!")

---
## 2. Data Loading <a name="data-loading"></a>

In [ ]:
# ============================================
# Cell 2: Load Data
# ============================================
DATA_DIR = '../data/raw/'
train_df = pd.read_csv(f'{DATA_DIR}train.csv')

# Compute composite target
train_df['composite_score'] = (
    train_df['humanity_score'] * 0.35 +
    train_df['performance_score'] * 0.25 +
    train_df['helpfulness_score'] * 0.40
)

# Custom stop words (add domain-specific)
custom_stop_words = list(ENGLISH_STOP_WORDS) + [
    'also', 'would', 'could', 'should', 'however', 'therefore',
    ' furthermore', 'addition', 'example', 'include', 'including'
]

print(f"Dataset: {train_df.shape}")
print(f"Score range: {train_df['composite_score'].min():.2f} - {train_df['composite_score'].max():.2f}")

In [ ]:
# ============================================
# Cell 3: Text Preprocessing
# ============================================
def preprocess_text(text):
    """Clean and normalize text for TF-IDF."""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['clean_prompt'] = train_df['prompt'].apply(preprocess_text)
train_df['clean_response'] = train_df['response'].apply(preprocess_text)

print("Text preprocessed!")
print(f"Sample prompt:    {train_df['clean_prompt'].iloc[0][:100]}...")
print(f"Sample response:  {train_df['clean_response'].iloc[0][:100]}...")

---
## 3. TF-IDF on Prompts <a name="tfidf-prompts"></a>

In [ ]:
# ============================================
# Cell 4: Prompt TF-IDF
# ============================================
prompt_tfidf = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),
    stop_words=custom_stop_words,
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

prompt_features = prompt_tfidf.fit_transform(train_df['clean_prompt'])
print(f"Prompt TF-IDF matrix: {prompt_features.shape}")
print(f"Vocabulary size: {len(prompt_tfidf.vocabulary_)}")

# Top prompt terms
prompt_sum = np.array(prompt_features.sum(axis=0)).flatten()
top_prompt_idx = prompt_sum.argsort()[::-1][:15]
feature_names = prompt_tfidf.get_feature_names_out()

print("\nTop 15 Prompt TF-IDF Terms:")
for idx in top_prompt_idx:
    print(f"  {feature_names[idx]:25s} {prompt_sum[idx]:.3f}")

---
## 4. TF-IDF on Responses <a name="tfidf-responses"></a>

In [ ]:
# ============================================
# Cell 5: Response TF-IDF
# ============================================
response_tfidf = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 3),
    stop_words=custom_stop_words,
    min_df=2,
    max_df=0.85,
    sublinear_tf=True
)

response_features = response_tfidf.fit_transform(train_df['clean_response'])
print(f"Response TF-IDF matrix: {response_features.shape}")

# Top response terms
resp_sum = np.array(response_features.sum(axis=0)).flatten()
top_resp_idx = resp_sum.argsort()[::-1][:15]
resp_feature_names = response_tfidf.get_feature_names_out()

print("\nTop 15 Response TF-IDF Terms:")
for idx in top_resp_idx:
    print(f"  {resp_feature_names[idx]:30s} {resp_sum[idx]:.3f}")

In [ ]:
# ============================================
# Cell 6: TF-IDF Visualization
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Prompt top terms
p_top = top_prompt_idx[:10]
p_words = [feature_names[i] for i in p_top]
p_vals = [prompt_sum[i] for i in p_top]
axes[0].barh(range(len(p_words)), p_vals, color='#3498db', alpha=0.8)
axes[0].set_yticks(range(len(p_words)))
axes[0].set_yticklabels(p_words)
axes[0].invert_yaxis()
axes[0].set_title('Top Prompt TF-IDF Terms', fontsize=12, fontweight='bold')

# Response top terms
r_top = top_resp_idx[:10]
r_words = [resp_feature_names[i] for i in r_top]
r_vals = [resp_sum[i] for i in r_top]
axes[1].barh(range(len(r_words)), r_vals, color='#e74c3c', alpha=0.8)
axes[1].set_yticks(range(len(r_words)))
axes[1].set_yticklabels(r_words)
axes[1].invert_yaxis()
axes[1].set_title('Top Response TF-IDF Terms', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/tfidf_top_terms.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Combined TF-IDF Features <a name="combined-tfidf"></a>

In [ ]:
# ============================================
# Cell 7: Dimensionality Reduction with SVD
# ============================================
from scipy.sparse import hstack

# Combine prompt + response TF-IDF
combined_tfidf = hstack([prompt_features, response_features])
print(f"Combined TF-IDF: {combined_tfidf.shape}")

# Reduce dimensionality
svd = TruncatedSVD(n_components=100, random_state=42)
X_svd = svd.fit_transform(combined_tfidf)

print(f"\nSVD reduced: {X_svd.shape}")
print(f"Explained variance (top 100): {svd.explained_variance_ratio_.sum():.3f}")

# Scree plot
fig, ax = plt.subplots(figsize=(10, 5))
cumulative = np.cumsum(svd.explained_variance_ratio_)
ax.plot(range(1, len(cumulative)+1), cumulative, 'b-o', markersize=4, linewidth=1.5)
ax.axhline(0.80, color='red', linestyle='--', alpha=0.5, label='80% variance')
ax.set_xlabel('Number of Components', fontsize=12)
ax.set_ylabel('Cumulative Explained Variance', fontsize=12)
ax.set_title('SVD Scree Plot - TF-IDF Features', fontsize=14, fontweight='bold')
ax.legend(loc='best')
plt.tight_layout()
plt.savefig('../data/processed/tfidf_scree.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. N-Gram Analysis <a name="ngram-analysis"></a>

In [ ]:
# ============================================
# Cell 8: N-Gram Analysis (Bigrams & Trigrams)
# ============================================
# Bigrams
bigram_tfidf = TfidfVectorizer(
    ngram_range=(2, 2), max_features=200, stop_words=custom_stop_words,
    min_df=2, sublinear_tf=True
)
bigram_features = bigram_tfidf.fit_transform(train_df['clean_response'])

# Trigrams
trigram_tfidf = TfidfVectorizer(
    ngram_range=(3, 3), max_features=100, stop_words=custom_stop_words,
    min_df=2, sublinear_tf=True
)
trigram_features = trigram_tfidf.fit_transform(train_df['clean_response'])

print(f"Bigram features: {bigram_features.shape}")
print(f"Trigram features: {trigram_features.shape}")

# Top bigrams
bg_sum = np.array(bigram_features.sum(axis=0)).flatten()
bg_names = bigram_tfidf.get_feature_names_out()
top_bg = bg_sum.argsort()[::-1][:10]

print("\nTop 10 Bigrams:")
for i in top_bg:
    print(f"  {bg_names[i]:35s} {bg_sum[i]:.3f}")

---
## 7. Model Training with TF-IDF <a name="model-training"></a>

In [ ]:
# ============================================
# Cell 9: Train/Test Split
# ============================================
y = train_df['composite_score']

X_train, X_val, y_train, y_val = train_test_split(
    X_svd, y, test_size=0.2, random_state=42
)

print(f"Training: {X_train.shape}")
print(f"Validation: {X_val.shape}")

In [ ]:
# ============================================
# Cell 10: Model Training
# ============================================
models = {
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'SVR (RBF)': SVR(kernel='rbf', C=10)
}

tfidf_results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    tfidf_results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f"{name:25s} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}")

---
## 8. Feature Importance <a name="feature-importance"></a>

In [ ]:
# ============================================
# Cell 11: Feature Importance from Ridge
# ============================================
ridge = models['Ridge']
coefficients = ridge.coef_

top_positive = np.argsort(coefficients)[::-1][:10]
top_negative = np.argsort(coefficients)[:10]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Positive features
pos_idx = top_positive[:10]
axes[0].barh(range(len(pos_idx)), coefficients[pos_idx], color='#2ecc71', alpha=0.85)
axes[0].set_yticks(range(len(pos_idx)))
axes[0].set_yticklabels([f'SVD-{i}' for i in pos_idx])
axes[0].set_title('Top 10 Positive Features', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Coefficient')

# Negative features
neg_idx = top_negative[:10]
axes[1].barh(range(len(neg_idx)), coefficients[neg_idx], color='#e74c3c', alpha=0.85)
axes[1].set_yticks(range(len(neg_idx)))
axes[1].set_yticklabels([f'SVD-{i}' for i in neg_idx])
axes[1].set_title('Top 10 Negative Features', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Coefficient')

plt.tight_layout()
plt.savefig('../data/processed/tfidf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Results Comparison <a name="results-comparison"></a>

In [ ]:
# ============================================
# Cell 12: Final Results
# ============================================
results_df = pd.DataFrame(tfidf_results).T

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results_df))
width = 0.25

ax.bar(x - width, results_df['RMSE'], width, label='RMSE', color='#e74c3c', alpha=0.8)
ax.bar(x, results_df['MAE'], width, label='MAE', color='#3498db', alpha=0.8)
ax.bar(x + width, results_df['R2'], width, label='R2', color='#2ecc71', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(results_df.index, rotation=30, ha='right', fontsize=9)
ax.set_title('TF-IDF Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='best')

plt.tight_layout()
plt.savefig('../data/processed/tfidf_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nAll Results:")
print(results_df.round(4).to_string())

# Save
results_df.to_csv('../data/processed/tfidf_experiment_results.csv')
print("\nResults saved!")

---
## ✅ TF-IDF Experiment Complete

**Key Takeaways:**
- TF-IDF captures semantic patterns correlated with quality
- Response features are more informative than prompt features
- SVD dimensionality reduction maintains performance while reducing features

**Next:** Combine with baseline features in `gemma_4_good_final.ipynb`